# 04 — Modelos Locais com Ollama

**Módulo:** EAI_07 — IA Generativa  
**Submódulo:** 02_Modelos_PreTreinados

---

## O que você vai aprender

- O que é o **Ollama** e por que usar modelos locais
- Como **instalar e configurar** o Ollama
- Como **baixar e gerenciar modelos** localmente
- Como usar modelos locais com o **mesmo `llm_factory.py`** do projeto
- **Comparativo prático**: DeepSeek (API) vs Ollama (local)
- Quando usar cada abordagem

---

> 💡 **Por que modelos locais?**  
> - Gratuito — sem custo de API  
> - Privado — dados não saem da sua máquina  
> - Offline — funciona sem internet  
> - Limitação: depende do hardware disponível

## Conceito: como o Ollama funciona?

```
┌─────────────────────────────────────────────────────────┐
│                    Seu Computador                       │
│                                                         │
│   Jupyter Notebook                                      │
│        │                                                │
│        │ HTTP (mesma interface do OpenAI SDK)           │
│        ↓                                                │
│   Ollama Server  ←── roda em background                 │
│   localhost:11434    (como um mini-servidor local)      │
│        │                                                │
│        ↓                                                │
│   Modelo (.gguf)  ←── arquivo baixado (~2GB)            │
│   ex: llama3.2, mistral, phi3                           │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

O Ollama expõe uma **API compatível com OpenAI** — por isso o `llm_factory.py`  
funciona sem nenhuma alteração, apenas mudando o `.env`.

---
## 1. Instalação do Ollama

Execute os passos abaixo **fora do notebook** (no terminal do seu sistema).

### Windows
1. Acesse [ollama.com/download](https://ollama.com/download)
2. Baixe o instalador `.exe`
3. Execute o instalador — o Ollama inicia automaticamente como serviço

### Linux
```bash
curl -fsSL https://ollama.com/install.sh | sh
```

### macOS
```bash
brew install ollama
```

---

### Verificar se está rodando
```bash
# No terminal — deve retornar "Ollama is running"
curl http://localhost:11434
```

---

### Modelos recomendados para 8GB RAM

| Modelo | Tamanho | Velocidade | Qualidade | Comando |
|--------|---------|------------|-----------|----------|
| `llama3.2` | ~2GB | ⚡ Rápido | ⭐⭐⭐ | `ollama pull llama3.2` |
| `phi3` | ~2.3GB | ⚡ Rápido | ⭐⭐⭐ | `ollama pull phi3` |
| `mistral` | ~4GB | 🐢 Médio | ⭐⭐⭐⭐ | `ollama pull mistral` |
| `llama3.2:1b` | ~1.3GB | ⚡⚡ Muito rápido | ⭐⭐ | `ollama pull llama3.2:1b` |

> **Para 8GB RAM:** comece com `llama3.2` ou `phi3`.

---
## 2. Verificando o Ollama via Python

In [1]:
import requests

def verificar_ollama(base_url: str = "http://localhost:11434") -> bool:
    """Verifica se o servidor Ollama está rodando."""
    try:
        r = requests.get(base_url, timeout=3)
        print(f"✅ Ollama está rodando em {base_url}")
        return True
    except requests.exceptions.ConnectionError:
        print(f"❌ Ollama não encontrado em {base_url}")
        print("   → Inicie o Ollama antes de continuar")
        print("   → Windows: abra o app Ollama na bandeja do sistema")
        print("   → Linux/Mac: execute 'ollama serve' no terminal")
        return False

ollama_disponivel = verificar_ollama()

✅ Ollama está rodando em http://localhost:11434


In [3]:
def listar_modelos_ollama(base_url: str = "http://localhost:11434") -> list:
    """Lista os modelos disponíveis localmente no Ollama."""
    try:
        r = requests.get(f"{base_url}/api/tags", timeout=5)
        modelos = r.json().get("models", [])

        if not modelos:
            print("Nenhum modelo baixado ainda.")
            print("Execute no terminal: ollama pull llama3.2")
            return []

        print(f"{len(modelos)} modelo(s) disponível(is) localmente:\n")
        for m in modelos:
            tamanho_gb = m.get('size', 0) / 1e9
            print(f"  📦 {m['name']:<30} {tamanho_gb:.1f} GB")

        return [m['name'] for m in modelos]

    except Exception as e:
        print(f"Erro ao listar modelos: {e}")
        return []

if ollama_disponivel:
    modelos_locais = listar_modelos_ollama()

1 modelo(s) disponível(is) localmente:

  📦 llama3.2:latest                2.0 GB


---
## 3. Usando Ollama com o llm_factory

Basta mudar duas linhas no `.env` — o resto do código não muda.

In [4]:
# Para usar Ollama, seu .env deve ter:
#   LLM_PROVIDER=ollama
#   LLM_MODEL=llama3.2
#
# Aqui vamos demonstrar as duas formas:
# A) Usando o llm_factory (lê o .env)
# B) Chamada direta ao Ollama via SDK OpenAI

import sys, os
sys.path.append(os.path.abspath('..'))

from shared.llm_factory import get_provider_info

info = get_provider_info()
print(f"Provider atual: {info['provider']}")
print(f"Modelo atual  : {info['model']}")
print()
print("Para testar o Ollama sem mudar o .env,")
print("vamos usar o SDK OpenAI apontando direto para localhost.")

Provider atual: ollama
Modelo atual  : llama3.2

Para testar o Ollama sem mudar o .env,
vamos usar o SDK OpenAI apontando direto para localhost.


In [5]:
from openai import OpenAI

def chat_ollama(prompt: str, modelo: str = "llama3.2", system: str = None) -> str:
    """
    Chama o Ollama local diretamente.
    Usa a mesma interface do SDK OpenAI — só muda a base_url.
    """
    client = OpenAI(
        api_key="ollama",                      # Ollama não usa chave real
        base_url="http://localhost:11434/v1"   # Servidor local
    )

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=modelo,
        messages=messages,
        temperature=0.2
    )
    return response.choices[0].message.content


if ollama_disponivel and modelos_locais:
    modelo_teste = modelos_locais[0]  # usa o primeiro modelo disponível
    print(f"Testando com modelo: {modelo_teste}\n")

    resposta = chat_ollama(
        prompt="O que é um Large Language Model? Responda em 3 linhas.",
        modelo=modelo_teste
    )
    print(resposta)
else:
    print("Ollama não disponível — execute as células anteriores após instalar.")

Testando com modelo: llama3.2:latest

Um Large Language Model (LLM) é um tipo de modelo de linguagem artificial projetado para processar e gerar grande volumes de texto com alta precisão. Esses modelos são treinados por grandes quantidades de dados textuais, o que permite que eles aprendam padrões e estruturas na língua humana. Eles são usados em diversas aplicações, como tradução automática, respostas de perguntas e geração de conteúdo.


---
## 4. Comparativo: DeepSeek API vs Ollama Local

Vamos medir velocidade e qualidade de resposta para o mesmo prompt.

In [6]:
import time
from dotenv import load_dotenv
load_dotenv('../.env')

PROMPT_TESTE = "Explique o conceito de overfitting em machine learning em 3 linhas."

def medir_resposta(nome: str, fn_chat, prompt: str) -> dict:
    """Mede tempo e captura resposta de uma função de chat."""
    inicio = time.time()
    try:
        resposta = fn_chat(prompt)
        tempo = time.time() - inicio
        return {"nome": nome, "tempo": tempo, "resposta": resposta, "erro": None}
    except Exception as e:
        return {"nome": nome, "tempo": 0, "resposta": "", "erro": str(e)}


# ── DeepSeek via API ─────────────────────────────────────────
def chat_deepseek(prompt):
    client = OpenAI(
        api_key=os.getenv("DEEPSEEK_API_KEY"),
        base_url="https://api.deepseek.com"
    )
    r = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2, max_tokens=200
    )
    return r.choices[0].message.content


# ── Teste DeepSeek ───────────────────────────────────────────
print("Testando DeepSeek...")
resultado_deepseek = medir_resposta("DeepSeek (API)", chat_deepseek, PROMPT_TESTE)

print(f"\n{'='*55}")
print(f"Provider : {resultado_deepseek['nome']}")
print(f"Tempo    : {resultado_deepseek['tempo']:.2f}s")
print(f"Resposta :\n{resultado_deepseek['resposta']}")

Testando DeepSeek...

Provider : DeepSeek (API)
Tempo    : 11.65s
Resposta :
Overfitting ocorre quando um modelo de machine learning se ajusta excessivamente aos dados de treinamento, capturando ruídos e padrões irrelevantes. Isso resulta em alta performance no conjunto de treinamento, mas baixa capacidade de generalização para dados novos. É como decorar as respostas de uma prova em vez de aprender o conceito.


In [7]:
# ── Teste Ollama ─────────────────────────────────────────────
if ollama_disponivel and modelos_locais:
    modelo_local = modelos_locais[0]

    def chat_ollama_fn(prompt):
        return chat_ollama(prompt, modelo=modelo_local)

    print(f"Testando Ollama ({modelo_local})...")
    resultado_ollama = medir_resposta(f"Ollama ({modelo_local})", chat_ollama_fn, PROMPT_TESTE)

    print(f"\n{'='*55}")
    print(f"Provider : {resultado_ollama['nome']}")
    print(f"Tempo    : {resultado_ollama['tempo']:.2f}s")
    print(f"Resposta :\n{resultado_ollama['resposta']}")

    # ── Resumo comparativo ───────────────────────────────────
    print(f"\n{'='*55}")
    print("RESUMO COMPARATIVO")
    print('='*55)
    print(f"{'Provider':<25} {'Tempo':>8} {'Custo'}")
    print('-'*55)
    print(f"{'DeepSeek (API)':<25} {resultado_deepseek['tempo']:>7.2f}s  ~$0.0001/chamada")
    print(f"{resultado_ollama['nome']:<25} {resultado_ollama['tempo']:>7.2f}s  Gratuito")
else:
    print("Ollama não disponível para comparação.")
    print("Instale e execute: ollama pull llama3.2")

Testando Ollama (llama3.2:latest)...

Provider : Ollama (llama3.2:latest)
Tempo    : 31.31s
Resposta :
O overfitting é um problema comum em machine learning que ocorre quando um modelo de aprendizado de máquina se adapta excessivamente às características específicas do conjunto de dados de treinamento, resultando em desempenho ruim em novos dados não vistos. Isso acontece porque o modelo está memorizando as noções específicas do conjunto de dados de treinamento e não pode generalizar bem para outros conjuntos de dados.

RESUMO COMPARATIVO
Provider                     Tempo Custo
-------------------------------------------------------
DeepSeek (API)              11.65s  ~$0.0001/chamada
Ollama (llama3.2:latest)    31.31s  Gratuito


---
## 5. Configurando o .env para usar Ollama

Para trocar permanentemente para o Ollama, edite o `.env` na raiz do EAI_07:

In [8]:
# Demonstra como seria o .env para cada provider

configs = {
    "DeepSeek (atual)": """
LLM_PROVIDER=deepseek
LLM_MODEL=deepseek-chat
DEEPSEEK_API_KEY=sk-...
""",
    "Ollama llama3.2": """
LLM_PROVIDER=ollama
LLM_MODEL=llama3.2
# Sem chave de API necessária
""",
    "Ollama mistral": """
LLM_PROVIDER=ollama
LLM_MODEL=mistral
# Sem chave de API necessária
""",
    "Anthropic Claude Haiku": """
LLM_PROVIDER=anthropic
LLM_MODEL=claude-haiku-4-5
ANTHROPIC_API_KEY=sk-ant-...
"""
}

for nome, config in configs.items():
    print(f"{'='*40}")
    print(f"# {nome}")
    print(config)

# DeepSeek (atual)

LLM_PROVIDER=deepseek
LLM_MODEL=deepseek-chat
DEEPSEEK_API_KEY=sk-...

# Ollama llama3.2

LLM_PROVIDER=ollama
LLM_MODEL=llama3.2
# Sem chave de API necessária

# Ollama mistral

LLM_PROVIDER=ollama
LLM_MODEL=mistral
# Sem chave de API necessária

# Anthropic Claude Haiku

LLM_PROVIDER=anthropic
LLM_MODEL=claude-haiku-4-5
ANTHROPIC_API_KEY=sk-ant-...



---
## 6. Gerenciando modelos no Ollama

Comandos úteis para o terminal:

In [9]:
# Você pode gerenciar modelos direto do Python via subprocess
import subprocess

def ollama_cmd(args: list) -> str:
    """Executa um comando ollama e retorna a saída."""
    try:
        result = subprocess.run(
            ["ollama"] + args,
            capture_output=True, text=True, timeout=10
        )
        return result.stdout or result.stderr
    except FileNotFoundError:
        return "Ollama não encontrado no PATH"
    except subprocess.TimeoutExpired:
        return "Timeout — operação muito lenta"

# Lista modelos instalados
print("Modelos instalados:")
print(ollama_cmd(["list"]))

print("\nComandos úteis (execute no terminal):")
comandos = [
    ("ollama list",              "Lista modelos instalados"),
    ("ollama pull llama3.2",     "Baixa o llama3.2 (~2GB)"),
    ("ollama pull phi3",         "Baixa o phi3 (~2.3GB)"),
    ("ollama pull mistral",      "Baixa o mistral (~4GB)"),
    ("ollama rm llama3.2",       "Remove o llama3.2"),
    ("ollama run llama3.2",      "Chat interativo no terminal"),
    ("ollama serve",             "Inicia o servidor (Linux/Mac)"),
]
for cmd, desc in comandos:
    print(f"  {cmd:<35} # {desc}")

Modelos instalados:
NAME               ID              SIZE      MODIFIED      
llama3.2:latest    a80c4f17acd5    2.0 GB    8 minutes ago    


Comandos úteis (execute no terminal):
  ollama list                         # Lista modelos instalados
  ollama pull llama3.2                # Baixa o llama3.2 (~2GB)
  ollama pull phi3                    # Baixa o phi3 (~2.3GB)
  ollama pull mistral                 # Baixa o mistral (~4GB)
  ollama rm llama3.2                  # Remove o llama3.2
  ollama run llama3.2                 # Chat interativo no terminal
  ollama serve                        # Inicia o servidor (Linux/Mac)


---
## Resumo: quando usar cada provider?

| Situação | Provider recomendado |
|---|---|
| Desenvolvimento diário | **DeepSeek** — rápido e barato |
| Créditos acabando | **Ollama** — fallback gratuito |
| Dados confidenciais | **Ollama** — nada sai da máquina |
| Sem internet | **Ollama** — 100% local |
| Melhor qualidade | **DeepSeek-Reasoner** ou **Claude Sonnet** |
| Testes rápidos | **Ollama llama3.2:1b** — mais leve |

### Estratégia do projeto EAI_07

```
Desenvolvimento  →  DeepSeek (padrão no .env)
Sem créditos     →  Muda LLM_PROVIDER=ollama no .env
Notebooks didáticos → comparar os dois lado a lado
Assistente Técnico  → DeepSeek em produção
```

---